# Proyecto Final | Predicción de Readmisión Hospitalaria
## Notebook 1 — Carga y Exploración de Datos (EDA)

**Dataset:** Diabetes 130-US Hospitals (1999–2008) — UCI ML Repository  
**Objetivo:** Predecir si un paciente diabético será **readmitido en menos de 30 días** tras el alta  
**Tipo de problema:** Clasificación binaria  

---
### Contexto clínico
La readmisión hospitalaria en menos de 30 días es un indicador clave de calidad asistencial y supone un alto coste para el sistema sanitario. Este dataset recoge 10 años de registros clínicos de pacientes diabéticos en 130 hospitales de EE.UU. El objetivo es identificar qué factores aumentan el riesgo de readmisión temprana para que los equipos médicos puedan intervenir de forma preventiva al alta.

## Tabla de Contenidos
1. [Importar librerías](#1)
2. [Cargar el dataset](#2)
3. [Exploración inicial](#3)
4. [Distribución del target](#4)
5. [Estadísticas descriptivas](#5)
6. [Análisis de valores faltantes](#6)
7. [Variables categóricas](#7)
8. [Variables numéricas](#8)
9. [Matriz de correlación](#9)
10. [Guardar datos limpios](#10)

## 1. Importar librerías <a id='1'></a>

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Utilidades
import os

# Configuración de visualización
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 55)

print('Librerías importadas correctamente')

## 2. Cargar el dataset <a id='2'></a>

**Diabetes 130-US Hospitals for years 1999–2008**  
Fuente: UCI Machine Learning Repository  

| Característica | Valor |
|----------------|-------|
| Muestras | 101.766 |
| Features | 49 (numéricas + categóricas) |
| Target | `readmitted` — NO / >30 / <30 |
| Período | 1999–2008 |
| Hospitales | 130 hospitales de EE.UU. |

In [ ]:
url = 'https://raw.githubusercontent.com/14Richa/Patient-Readmission-Analysis/main/diabetic_data.csv'
df = pd.read_csv(url)

# Crear target binario: 1 = readmitido en <30 días, 0 = no readmitido en <30 días
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

print('Dataset cargado correctamente')
print(f'   {df.shape[0]:,} pacientes | {df.shape[1]} columnas')
df.head()

## 3. Exploración inicial <a id='3'></a>

In [ ]:
# Dimensiones del dataset
print('=== SHAPE ===')
print(f'Filas:    {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')

In [ ]:
# Tipos de datos — mezcla de numéricos y categóricos
print('=== TIPOS DE DATOS ===')
print(df.dtypes.value_counts())
print('\nDetalle:')
print(df.dtypes)

In [ ]:
# Valores únicos por columna — detecta columnas ID y columnas con pocos valores
print('=== VALORES ÚNICOS POR COLUMNA ===')
for col in df.columns:
    print(f'{col:40s}: {df[col].nunique():>6} únicos')

## 4. Distribución del target <a id='4'></a>

El target original tiene 3 clases: `NO`, `>30`, `<30`.  
Lo convertimos a **binario**: `readmitted_30 = 1` si el paciente fue readmitido en menos de 30 días.

In [ ]:
# Distribución del target original
print('=== TARGET ORIGINAL (3 clases) ===')
conteo_orig = df['readmitted'].value_counts()
pct_orig = df['readmitted'].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({'Cantidad': conteo_orig, 'Porcentaje (%)': pct_orig}))

print('\n=== TARGET BINARIO (readmitido en <30 días) ===')
conteo = df['readmitted_30'].value_counts()
pct = df['readmitted_30'].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({'Cantidad': conteo, 'Porcentaje (%)': pct}))

In [ ]:
# Visualización del target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target original
conteo_orig = df['readmitted'].value_counts()
axes[0].bar(conteo_orig.index, conteo_orig.values,
            color=['green', 'orange', 'red'], edgecolor='white', width=0.5)
axes[0].set(title='Distribución original del target',
            ylabel='Número de pacientes')
for i, v in enumerate(conteo_orig.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontweight='bold', fontsize=10)

# Target binario
conteo_bin = df['readmitted_30'].value_counts().sort_index()
labels = ['No readmitido <30 (0)', 'Readmitido <30 (1)']
axes[1].bar(labels, conteo_bin.values, color=['green', 'red'],
            edgecolor='white', width=0.5)
axes[1].set(title='Target binario — Readmisión en <30 días',
            ylabel='Número de pacientes')
for i, v in enumerate(conteo_bin.values):
    axes[1].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Predicción de Readmisión Hospitalaria — Variable Target', fontsize=14)
plt.tight_layout()
plt.show()

print('Dataset muy desbalanceado: solo ~11% de pacientes readmitidos en <30 días')
print('→ Usaremos F1-score, ROC-AUC y Recall como métricas principales')
print('→ Consideraremos class_weight="balanced" en los modelos')

## 5. Estadísticas descriptivas <a id='5'></a>

In [ ]:
# Variables numéricas
print('=== ESTADÍSTICAS DE VARIABLES NUMÉRICAS ===')
df.select_dtypes(include='number').describe().round(2)

In [ ]:
# Comparar medias por target — nos ayuda a ver qué variables difieren entre clases
print('=== MEDIAS POR DIAGNÓSTICO (0=No readmitido / 1=Readmitido <30) ===')
df.groupby('readmitted_30').mean(numeric_only=True).round(2).T

## 6. Análisis de valores faltantes <a id='6'></a>

Este dataset codifica los valores faltantes como `'?'` en lugar de `NaN`.  
Primero los convertimos para poder detectarlos correctamente.

In [ ]:
# Reemplazar '?' por NaN para que pandas los reconozca como valores faltantes
df.replace('?', np.nan, inplace=True)

# Calcular número y porcentaje de nulos por columna
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(1)
nulos_df = pd.DataFrame({'Nulos': nulos, '% Nulos': pct_nulos})
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('% Nulos', ascending=False)

print('Columnas con valores faltantes:')
print(nulos_df)
print(f'\nTotal de celdas con NaN: {nulos.sum():,}')

In [ ]:
# Visualización de nulos — coloreamos según severidad
nulos_plot = nulos_df[nulos_df['% Nulos'] > 0]

plt.figure(figsize=(10, 5))
bars = plt.barh(nulos_plot.index, nulos_plot['% Nulos'],
                color=['red' if v > 40 else 'orange' if v > 10 else 'steelblue'
                       for v in nulos_plot['% Nulos']])
plt.axvline(x=40, color='red', linestyle='--', alpha=0.7, label='Umbral 40% → eliminar')
plt.xlabel('% Valores faltantes')
plt.title('Valores faltantes por columna')
plt.legend()
plt.tight_layout()
plt.show()

cols_eliminar = nulos_df[nulos_df['% Nulos'] > 40].index.tolist()
print(f'\nColumnas a eliminar (>40% nulos): {cols_eliminar}')
print('→ Se tratarán en el Notebook 2')

## 7. Variables categóricas <a id='7'></a>

In [ ]:
# Distribución de las principales variables categóricas
cat_cols = ['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id']

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    conteo = df[col].value_counts().head(10)
    axes[i].barh(conteo.index.astype(str)[::-1], conteo.values[::-1],
                 color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=11, fontweight='bold')

axes[5].axis('off')
plt.suptitle('Distribución de variables categóricas principales', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de readmisión por grupo de edad
tasa_edad = df.groupby('age')['readmitted_30'].mean().sort_index().mul(100).round(1)

plt.figure(figsize=(10, 5))
tasa_edad.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Tasa de readmisión (<30 días) por grupo de edad')
plt.xlabel('Grupo de edad')
plt.ylabel('Tasa de readmisión (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Variables numéricas <a id='8'></a>

In [ ]:
# Histogramas de variables numéricas por target
# Nos ayuda a ver si las distribuciones difieren entre clases
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
            'num_medications', 'number_diagnoses', 'number_inpatient']

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for val, color, label in zip([0, 1], ['green', 'red'],
                                  ['No readmitido', 'Readmitido <30']):
        axes[i].hist(df[df['readmitted_30'] == val][col].dropna(),
                     bins=25, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución de variables numéricas por target', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Matriz de correlación <a id='9'></a>

In [ ]:
# Correlación entre variables numéricas
num_df = df.select_dtypes(include='number')
corr = num_df.corr().round(2)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)),
            cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.3)
plt.title('Matriz de Correlación — Variables numéricas', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Top features más correlacionadas con el target
corr_target = corr['readmitted_30'].drop('readmitted_30').abs().sort_values(ascending=False)

print('Top features numéricas correlacionadas con readmitted_30:')
print(corr_target.head(10).round(3).to_string())

## 10. Guardar datos limpios <a id='10'></a>

In [ ]:
# Guardamos el dataset con los '?' ya convertidos a NaN para usarlo en el Notebook 2
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/readmision_raw.csv', index=False)

print('Dataset guardado en ../data/readmision_raw.csv')
print(f'Shape: {df.shape}')
print(f'Valores nulos: {df.isnull().sum().sum():,}')

---
## Resumen del Notebook 1

| Aspecto | Resultado |
|---------|----------|
| **Muestras** | 101.766 pacientes |
| **Features** | 49 (numéricas + categóricas) |
| **Target** | Readmisión en <30 días (binario) |
| **Balance** | Muy desbalanceado — 11% positivos |
| **Valores nulos** | Codificados como `?` → convertidos a NaN |
| **Columnas a eliminar** | `weight`, `payer_code`, `medical_specialty` (>40% nulos) |
| **Encoding necesario** | Sí — múltiples variables categóricas |
| **Normalización** | Sí — rangos distintos entre features numéricas |
| **Métrica prioritaria** | ROC-AUC + Recall (clase positiva) |

---
**→ Próximo paso:** Notebook 2 — Limpieza, Encoding y Feature Engineering